<a href="https://colab.research.google.com/github/EsatovinVictoria/ML_Semester5_EsatovinEbenhaezerVictoria/blob/main/Pertemuan_6/Praktikum_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 4

Percobaan kali ini kita akan melihat perbedaan ketiga model yang telah kita bahas dan bandingkan hasilnya.

In [1]:
%pip install --no-cache-dir numpy pandas annoy faiss-cpu hnswlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib
from sklearn.preprocessing import StandardScaler

# 1. Dataset PBL (50.000 data, 5 fitur utama profil burnout)
df = pd.read_csv('burnout_dataset1.csv')
features_5d = ['Work_Hours_Per_Week', 'Sleep_Hours', 'Anxiety_Score', 'Depression_Score', 'Burnout_Score']
df_5d = df[features_5d].fillna(df[features_5d].mean(numeric_only=True))

scaler = StandardScaler()
X = scaler.fit_transform(df_5d).astype(np.float32)
n_data, dim = X.shape

query = X[0:1].copy()
k = 10
hasil = []

# 2. Annoy (Euclidean)
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")
hasil.append(["Euclidean", "Annoy", build_time, query_time, list(neighbors[0][:5])])

# 3. FAISS (IndexFlatL2)
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")
hasil.append(["Euclidean", "FAISS", build_time, query_time, list(indices[0][:5])])

# 4. HNSW (hnswlib)
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")
hasil.append(["Euclidean", "HNSW", build_time, query_time, list(labels[0][:5])])

=== Annoy ===


Build time: 0.23701977729797363 detik
Query time: 0.00028061866760253906 detik
Neighbors: [0, 17860, 20254, 32429, 33975] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.0011491775512695312 detik
Query time: 0.0034542083740234375 detik
Neighbors: [    0 17860 20254 32429 33975] ...

=== HNSW (hnswlib) ===


Build time: 0.45250749588012695 detik
Query time: 0.0003256797790527344 detik
Neighbors: [    0 17860 20254 32429 33975] ...


Lakukan percobaan pada metric distance yang berbeda. catat hasilnya pada tabel yang anda buat sendiri seperti pada praktikum 1.

### Percobaan Cosine

In [3]:
# 2. Annoy (Cosine / Angular)
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'angular')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")
hasil.append(["Cosine", "Annoy", build_time, query_time, list(neighbors[0][:5])])

# 3. FAISS (Cosine)
print("\n=== FAISS (Cosine) ===")
X_normalized = X.copy()
query_normalized = query.copy()
faiss.normalize_L2(X_normalized)
faiss.normalize_L2(query_normalized)
faiss_index = faiss.IndexFlatIP(dim)

start = time.time()
faiss_index.add(X_normalized)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query_normalized, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")
hasil.append(["Cosine", "FAISS", build_time, query_time, list(indices[0][:5])])

# 4. HNSW (hnswlib Cosine)
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='cosine', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")
hasil.append(["Cosine", "HNSW", build_time, query_time, list(labels[0][:5])])

=== Annoy ===


Build time: 0.3395206928253174 detik
Query time: 0.000339508056640625 detik
Neighbors: [0, 35296, 35143, 12261, 32429] ...

=== FAISS (Cosine) ===
Build time: 0.000438690185546875 detik
Query time: 0.0005242824554443359 detik
Neighbors: [    0 35296 35143 12261 32429] ...

=== HNSW (hnswlib) ===


Build time: 0.428652286529541 detik
Query time: 0.00023865699768066406 detik
Neighbors: [    0 35296 35143 12261 32429] ...


### Tabel Hasil Percobaan

In [4]:
import pandas as pd

pd.DataFrame(hasil, columns=[
    "Metrik", "Model", "Build Time (detik)", "Query Time (detik)", "5 Neighbors"
])

,Metrik,Model,Build Time (detik),Query Time (detik),5 Neighbors
0,Euclidean,Annoy,0.237020,0.000281,"[0, 17860, 20254, 32429, 33975]"
1,Euclidean,FAISS,0.001149,0.003454,"[0, 17860, 20254, 32429, 33975]"
2,Euclidean,HNSW,0.452507,0.000326,"[0, 17860, 20254, 32429, 33975]"
3,Cosine,Annoy,0.339521,0.000340,"[0, 35296, 35143, 12261, 32429]"
4,Cosine,FAISS,0.000439,0.000524,"[0, 35296, 35143, 12261, 32429]"
5,Cosine,HNSW,0.428652,0.000239,"[0, 35296, 35143, 12261, 32429]"
